Report 5


Anh Do

020416-2317

anhd@kth.se

In [209]:
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import numpy as np
import random
import itertools
from gurobipy import GRB
import gurobipy as gp
from pyomo.opt import SolverFactory

WLS = { # Please dont steal my credentials
    "WLSACCESSID": "1e6bdedb-f27d-4b0c-8a0d-24a8c94a8cb5",
    "WLSSECRET": "47ebeda5-b872-4365-9981-ef8044c28db5",
    "LICENSEID": 2707350,
}

# Problem 1

## Code

### ECP

In [ ]:
def build_ecp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [211]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as solver:
    solver.options.update(grb_params)
    
    model = build_ecp_model()
    solver.set_instance(model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Violation':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # 1. Solve Master
        solver.solve(model)
        
        # 2. Get current values
        x_val = {i: pyo.value(model.x[i]) for i in model.I}
        current_obj = pyo.value(model.obj)
        
        # 3. Check Constraint Violation: sum(x^2) - 3 <= 0
        sum_sq = sum(x_val[i]**2 for i in model.I)
        g_val = sum_sq - 3
        
        if g_val <= tol:
            print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f} (Converged!)")
            break
        
        print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f}")
        
        # 4. Add Cut: g(x^k) + grad * (x - x^k) <= 0
        # g(x^k) = g_val
        # grad = 2 * x_val
        lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
        
        model.ecp_cuts.add(lhs <= 0)
        solver.add_constraint(model.ecp_cuts[len(model.ecp_cuts)])

    print("-" * 40)
    print("Final Solution:")
    for i in model.I:
        print(f"x[{i}] = {pyo.value(model.x[i]):.6f}")

Iter  | Obj Value    | Violation   
----------------------------------------
1     | -10.000000   | 17.000000   


2     | -9.750000    | 24.062500   
3     | -8.257812    | 11.113464   
4     | -7.757812    | 12.090027   
5     | -7.615594    | 10.688251   
6     | -7.137504    | 11.594584   
7     | -7.035472    | 5.694672    
8     | -6.991201    | 7.421451    
9     | -6.419430    | 3.967551    
10    | -6.014282    | 4.314428    
11    | -6.000000    | 6.797126    
12    | -5.910535    | 4.113303    
13    | -5.876939    | 3.069151    
14    | -5.834757    | 5.812165    
15    | -5.647748    | 2.644989    
16    | -5.636855    | 2.761963    
17    | -5.469905    | 3.638892    
18    | -5.429711    | 2.471904    
19    | -5.386186    | 3.640453    
20    | -5.333902    | 1.434254    
21    | -5.317312    | 1.698371    
22    | -5.270109    | 4.099802    
23    | -5.240967    | 2.060639    
24    | -5.183710    | 1.908294    
25    | -4.989174    | 2.106066    
26    | -4.913810    | 1.410241    
27    | -4.908167    | 2.179136    
28    | -4.868768    | 1.547744    
29    | -4.845158    | 1.483

### OA

In [ ]:
# ==========================================================
# 1. OA Model
# ==========================================================

def create_nlp_model(y_fixed):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: 
            return pyo.Integers
        return pyo.Reals
    
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))
    
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    for i in range(5, 9):
        m.x[i].fix(y_fixed[i])

    # Constraints
    m.lincon = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    m.concon = pyo.Constraint(expr=sum(m.x[i]**2 for i in m.I) - 3 <= 0)

    return m

def create_feas_model(y_fixed):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: 
            return pyo.Integers
        return pyo.Reals
    
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))

    m.u = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.u, sense=pyo.minimize)
    
    for i in range(5, 9):
        m.x[i].fix(y_fixed[i])
        
    # Relaxed Constraints
    m.lincon = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= m.u)
    m.concon = pyo.Constraint(expr=sum(m.x[i]**2 for i in m.I) - 3 <= m.u)
    return m

def create_master_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: return pyo.Integers
        return pyo.Reals

        
    m.mu = pyo.Var(domain=pyo.Reals)
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))
    
    # Objective: Minimize mu
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)

    # My lazy cut so i dont need to handle vectorized constraints that is always the strongest (static cut) because we have linearity
    m.static_cut = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    
    m.feas_cuts = pyo.ConstraintList()
    m.ubd_cuts = pyo.ConstraintList()
    m.infeas_cuts = pyo.ConstraintList()
    return m

In [285]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as master_solver:
    master_solver.options.update(grb_params)
    master_model = create_master_model()
    master_solver.set_instance(master_model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0
    guess = {i: 0 for i in range(5, 9)} # Initial Integer Guess

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Status':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # --- 1. Solve NLP Subproblem ---
        NLP_model = create_nlp_model(guess)
        NLP_solver = SolverFactory('gurobi_direct')
        result_nlp = NLP_solver.solve(NLP_model, load_solutions=False)

        if result_nlp.solver.termination_condition == pyo.TerminationCondition.optimal:
            NLP_model.solutions.load_from(result_nlp)
            f_k = pyo.value(NLP_model.obj)
            x_k = np.array([pyo.value(NLP_model.x[i]) for i in master_model.I])

            # Update Primal Bound
            current_ubd = master_model.mu.ub if master_model.mu.has_ub() else float('inf')
            if f_k < current_ubd:
                master_model.mu.setub(f_k)
            
            print(f"{iteration:<5} | {f_k:<12.4f} | {'Optimal':<12}")

            # Add Optimality Cut
            master_model.ubd_cuts.add(
                # f_k + ∇f(x^k)ᵀ (x - x^k)
                expr=master_model.mu >= f_k + (-sum(master_model.x[i] - x_k[i] for i in master_model.I))
            )
            NLP_model.concon.pprint()
            # Add Feasibility Cut
            master_model.feas_cuts.add(
                # g_k + ∇g(x^k)ᵀ (x - x^k)
                expr= 0 >=  g_k
            )

        elif result_nlp.solver.termination_condition == pyo.TerminationCondition.infeasible:
            print(f"{iteration:<5} | {'--':<12} | {'Infeasible':<12}")

            # --- 2. Solve Feasibility Problem (Relaxation) ---
            FEAS_model = create_feas_model(guess)
            FEAS_solver = SolverFactory('gurobi_direct')
            FEAS_solver.solve(FEAS_model) # Assumed always feasible (min violation)
            
            # Add Feasibility Cut (from Lemma 1)
            new_cut = add_oa_cut(master_model, FEAS_model, guess, is_feasibility=True)
            if new_cut: master_solver.add_constraint(new_cut)

        else:
            raise RuntimeError(f"Unexpected NLP status: {result_nlp.solver.termination_condition}")

        # --- 3. Solve Master Problem ---
        result_master = master_solver.solve()

        if result_master.solver.termination_condition == TerminationCondition.infeasible:
            print("Master Infeasible -> Global Optimum Found.")
            break
        
        # --- 4. Update Integer Guess ---
        # Rounding is critical for integer vars returned by solvers
        guess = {i: int(round(pyo.value(master_model.y[i]))) for i in guess}

Iter  | Obj Value    | Status      
----------------------------------------
1     | -3.4641      | Optimal     


AttributeError: 'ConcreteModel' object has no attribute 'x'